# Classical models

$\renewcommand{\ket}[1]{\left|#1\right\rangle}\renewcommand{\bra}[1]{\left\langle #1\right|}\renewcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle}\renewcommand{\ketbra}[2]{\left|#1\right\rangle!\left\langle #2\right|}$This notebook shows how to convert the number of classical MCMC queries, for both the uniform move and the local spin-flip move, into an actual runtime estimate.

We are interested in finding the best classical baseline. For this reason, we benchmark heterogeneous platforms:

* A **sequential processor**. This processor runs at high clock frequency and has highly optimized floating-point arithmetic, but it offers limited flexibility for parallelizing the energy-difference computation. Therefore, the list of changed terms must be processed mostly sequentially. This corresponds to a single-core, high-performance CPU system with a large cache.

* A **fully parallel processor**. This processor allows us to design the energy-difference computation as a combinatorial circuit, exposing the available parallelism directly in hardware. The depth can scale logarithmically in the number of summed terms rather than linearly, but this flexibility comes with lower clock frequencies and larger prefactors. This corresponds to an FPGA-based system.

* A **throughput-oriented processor**. This processor is not ideal for accelerating a single MCMC trajectory, because the Markov-chain time direction remains sequential. However, it can parallelize part of the arithmetic. This corresponds to a GPU-based system. The time for one step of one chain may be comparable to, or even worse than, the CPU implementation because of branching, synchronization, and less efficient scalar control flow. The advantage of the GPU is throughput: when many independent chains or instances must be simulated, the relevant metric is not the latency of one chain, but the total wall-clock time required to process all chains.

We select high-end but realistic examples of these processors. The CPU-based system uses an Intel Xeon processor from a node of the CINECA Leonardo DCGP supercomputer. The GPU-based system uses an NVIDIA H100 NVL GPU. The FPGA-based system uses an AMD Virtex UltraScale+ XCVU19P (-2FSVA3824E).

We estimate the time per classical operation as follows.

1. For the CPU-based system, we use a C++ benchmark compiled with aggressive optimization flags for the target architecture. For each tested spin size $n$, we generate several SK instances and run a fixed number of Metropolis steps. The time per operation is obtained by dividing the total measured time by the total number of attempted moves. We then fit the following forms:

   * $P_\text{local}^\text{CPU}(n) = a_\ell + b_\ell n$ for the local spin-flip move,
   * $P_\text{unif}^\text{CPU}(n) = a_u + b_u n + c_u n^2$ for the uniform move.

   The difference comes from the cost of the energy-difference computation. A local spin flip changes only the terms involving the flipped spin, so its energy difference requires $O(n)$ work. A uniform move can change all spin values, so the full dense SK energy must be recomputed, requiring $O(n^2)$ work. The constant terms are interpreted as setup and loop-overhead contributions; when extrapolating the asymptotic per-query cost, we retain only the scaling-dependent part.

2. For the GPU-based system, we use a CUDA/C++ benchmark implementing the same dense SK Metropolis kernels. The GPU is not expected to reduce the sequential latency of a single chain substantially, but it can process many chains or model instances in parallel. Therefore, the relevant quantity is the achieved throughput across many independent trajectories, rather than only the latency of one trajectory. We then fit the following forms:

   * $P_\text{local}^\text{GPU}(n) = a_\ell + b_\ell n$ for the local spin-flip move,
   * $P_\text{unif}^\text{GPU}(n) = a_u + b_u n + c_u n^2$ for the uniform move.

3. For the FPGA-based system, we write C/C++ HLS kernels and synthesize them with AMD Vitis HLS into combinatorial circuits mapped to the target FPGA. Even on the high-end FPGA considered here, synthesis is feasible only up to moderate spin sizes before area becomes limiting. Therefore, we fit the values that can be synthesized and then project the timing to larger $n$, ignoring area constraints. This is analogous to the quantum-resource model used elsewhere in this work, where we count logical resources without requiring the resulting device to fit on a single physical chip. For both local and uniform moves, the fitted latency model has logarithmic form:

   * $P_\text{local}^\text{FPGA}(n) = a_\ell + b_\ell \log(n)$,
   * $P_\text{unif}^\text{FPGA}(n) = a_u + b_u \log(n)$.

The timing results are reported in the following notebooks and are used in the runtime plots in `4_runtime.ipynb`.

A main difference between the platforms is the arithmetic format. The CPU and GPU implementations use floating-point arithmetic, which is the natural and efficient choice on those devices. The FPGA implementation instead uses fixed-point arithmetic, so we must explicitly account for the discretization error introduced by finite precision.
